### Here we will find out all the important epitope with different filtering

In [1]:
import pandas as pd
import numpy as np
from Bio import SeqIO
from Bio.SeqRecord import SeqRecord
from Bio import AlignIO
import os
import pathlib

In [2]:
os.chdir("/home/kaderi/Feroj/thesis/HRV-C/Data_collecton/hrv-top5")

In [154]:
alignment_vp1 = AlignIO.read("vp4_aligned.fasta", "fasta")
print(alignment_vp1)
print(alignment_vp1.get_alignment_length())

Alignment with 5 rows and 66 columns
GAQVSKQNVGSHENSVSASGGSVIKYFNINYYKDSASSGLTKQD...ALM HRV-C2_VP4
GAQVSRQSVGSHETMIHAGTGAVVKYFNVNYYKDAASSGLTKQD...ALM HRV-C6_VP4
GAQVSRQKVGSHDNAINASSGSVVKYFNINYYKDAASSGLSKQD...ALM HRV-C9_VP4
GAQVSRQNNGTHESGVTANNGSVIKYFNINYYKDSASSGLSRQD...ALM HRV-C15_VP4
GAQVSKQSVGAHETTVHAGSGAVVKYFNINYYKDAASSGLTKQD...ALM HRV-C43_VP4
66


In [155]:
alignment_vp1[:,0]

'GGGGG'

In [156]:

consensus_sequence = ""
for residue in range(0,alignment_vp1.get_alignment_length()):
    unique_chars = ""
    for i in alignment_vp1[:,residue]:
        if i in unique_chars:
            pass
        else:
            unique_chars+=i
    char_dict = dict()

    for j in unique_chars:

        char_dict.update({f"{j}":alignment_vp1[:,residue].count(j)})


    keys = char_dict.keys()
    if '-' in keys:
        char_dict.pop("-")

    dict_value= list(char_dict.values())

    for keys, values in char_dict.items():
        if values == max(dict_value):
            consensus_sequence += keys
            break

                    

print(consensus_sequence)
print(len(consensus_sequence))
                




GAQVSRQNVGSHENSVHASSGSVVKYFNINYYKDAASSGLTKQDFSQDPSKFTQPVADVLTNPALM
66


In [157]:
conserved = ""
for i in range(alignment_vp1.get_alignment_length()):
    column = alignment_vp1[:, i]
    if column.count(column[0]) == len(column) and column[0] != "-":
        conserved += column[0]
    else:
        conserved += "-"

print("Conserved pattern:\n", conserved)
print(len(conserved))

Conserved pattern:
 GAQVS-Q--G-H-----A--G-V-KYFN-NYYKD-ASSGL--QDFSQDPSKFTQP----L-NPALM
66


In [158]:
## Here we will call the t-cell eptopes dataframe

tcel1_result = pd.read_csv(filepath_or_buffer="/home/kaderi/Feroj/thesis/HRV-C/Data_collecton/hrv-top5/result_collection/VP4_CTL/vp4_ctl_iedb_all_allele.csv")

In [159]:
# Now we keep to 1% of the peptide
one_percent = np.round(tcel1_result.shape[0]/100)

tcel1_result_filter_1 = tcel1_result.iloc[0:one_percent.astype(np.int32),:]
tcel1_result_filter_1

# Now we use second filtering if it's available, we will keep those peptide which percentile is less than 1
# we will write the programm if all the value less than one it will keep the data frame as it is, otherwise if will filter out
tcel1_result_filter_2 = tcel1_result[tcel1_result["netmhcpan_el percentile"] < 1]

### we will move to third filter where we will remove the duplicates

# tcel1_result_filter_3 = tcel1_result_filter_2.drop_duplicates(subset="peptide",ignore_index= True)
# tcel1_result_filter_2

In [160]:
peptide = []
new_allel= []
for i in tcel1_result_filter_2["peptide"].unique():
    allel_for_similar_epitope = tcel1_result_filter_2[tcel1_result_filter_2["peptide"]== i ]['allele'].values
    allels = ''
    if len(allel_for_similar_epitope)> 1:
        for j in allel_for_similar_epitope:
            allels += f"{j}"
            allels += ","
    else:
        allels += f"{allel_for_similar_epitope[0]}"
    peptide.append(i)
    new_allel.append(allels) 

print(new_allel)

['HLA-A*30:02,HLA-A*01:01,HLA-B*15:01,HLA-A*26:01,HLA-B*58:01,HLA-A*11:01,HLA-B*57:01,HLA-A*32:01,HLA-B*35:01,HLA-A*03:01,HLA-A*30:01,', 'HLA-A*31:01,HLA-A*30:01,HLA-A*33:01,HLA-A*30:02,HLA-A*11:01,HLA-A*23:01,HLA-A*03:01,', 'HLA-B*51:01,HLA-B*35:01,HLA-B*08:01,', 'HLA-A*30:02,HLA-B*15:01,HLA-A*26:01,HLA-A*32:01,HLA-A*30:01,', 'HLA-B*58:01,HLA-B*57:01,HLA-A*32:01,', 'HLA-A*68:01', 'HLA-A*26:01', 'HLA-A*30:02,HLA-B*15:01,', 'HLA-A*68:02,HLA-A*02:06,HLA-B*53:01,HLA-B*35:01,', 'HLA-A*68:01,HLA-A*30:01,HLA-A*11:01,HLA-A*03:01,', 'HLA-A*24:02,HLA-A*23:01,', 'HLA-B*40:01', 'HLA-A*68:02,HLA-A*02:03,', 'HLA-A*68:02']


In [161]:
tcel1_result_filter_3 = pd.DataFrame({'peptide':peptide,'allel':new_allel})
print(tcel1_result_filter_3.shape)


(14, 2)


#### Now we will match those peptides with conserve regions

In [162]:
# # for bepripred
# epitope_pred = ""

# for i,j in enumerate(bpred_result["EpitopeProbability"]):
#     if j < 0.5:
#         epitope_pred+="-"
#     else:
#         epitope_pred+=bpred_result['AminoAcid'][i]
# epitope_pred

In [163]:
# ## now we will cut the epitope separate then
# epitope_to_check= []
# for i in epitope_pred.split("-"):
#     if len(i)>=7:
#         epitope_to_check.append(i)
#     else:
#         pass
# epitope_to_check


In [164]:
epitope_to_check = tcel1_result_filter_3['peptide'].values
epitope_to_check

array(['ASSGSVVKY', 'KYFNINYYK', 'DPSKFTQPV', 'VVKYFNINY', 'SSGSVVKYF',
       'DAASSGLTK', 'DVLTNPALM', 'VKYFNINYY', 'FTQPVADVL', 'HASSGSVVK',
       'DFSQDPSKF', 'ADVLTNPAL', 'SVHASSGSV', 'NVGSHENSV'], dtype=object)

In [165]:
conserve_epitops= []
for i in conserved.split("-"):
    if len(i)>=5:
        conserve_epitops.append(i)
    else:
        pass
conserve_epitops

['GAQVS', 'NYYKD', 'ASSGL', 'QDFSQDPSKFTQP', 'NPALM']

In [166]:

result_table = pd.DataFrame(columns=("conserve_regions",
                                     'candidate_epitope',
                                     'conservence_presence'))
for a in conserve_epitops:
    l = ''
    k = ''
    for i in range(0,len(a)):

        b = a[i:]
        if b in l:
            break
        elif len(b)>=4:
            for epitopes in epitope_to_check:
                if b in epitopes:
                    print(f"{b} present in {epitopes}")
                    l += b
                    result_table_1 = pd.DataFrame({
                                        "conserve_regions":[a],
                                        "candidate_epitope":[epitopes],
                                        "conservence_presence":[l]})
                    result_table = pd.concat((result_table,result_table_1),axis = 0)
                    break



ASSGL present in DAASSGLTK
DPSKFTQP present in DPSKFTQPV
NPALM present in DVLTNPALM


In [167]:
allel_finder = result_table["candidate_epitope"].values.tolist()
allel_finder
allels_to_add = list()
for j in allel_finder:
    var_1 = tcel1_result_filter_3[tcel1_result_filter_3['peptide'] == j]
    allels_to_add.append(var_1['allel'].values[0])

    

result_table['allel'] = allels_to_add



In [168]:
conserve_regions_table = []
candidate_epitope_table = []
conserve_epitope_presence = []
allel_table = []
for i in result_table["conserve_regions"].unique():
    candidate_epitope_conserve = result_table[result_table["conserve_regions"]== i ]['candidate_epitope'].values
    presence_conserve = result_table[result_table["conserve_regions"]== i ]['conservence_presence'].values
    allels_unique = result_table[result_table["conserve_regions"]== i ]['allel'].values
    candidate_epitope_region_table = ''
    conserve_epitope_presence_table = ''
    allel_presence_table = ''
    if len(candidate_epitope_conserve)> 1:
        for j in candidate_epitope_conserve:
            candidate_epitope_region_table += f"{j}"
            candidate_epitope_region_table += ","
        for k in presence_conserve:
            conserve_epitope_presence_table += f"{k}"
            conserve_epitope_presence_table += ","
        for l in allels_unique:
            allel_presence_table += f"{l}"
            allel_presence_table += ","
    else:
        candidate_epitope_region_table += f"{candidate_epitope_conserve[0]}"
        conserve_epitope_presence_table += f"{presence_conserve[0]}"
        allel_presence_table += f"{allels_unique[0]}"

    conserve_regions_table.append(candidate_epitope_region_table)
    candidate_epitope_table.append(i)
    conserve_epitope_presence.append(conserve_epitope_presence_table)   
    allel_table.append(allel_presence_table)

final_table = pd.DataFrame({"conserve_region":candidate_epitope_table,
                            "epitope":conserve_regions_table,
                            "conservence_presence":conserve_epitope_presence,
                            "allel":allel_table
                            })
final_table

,conserve_region,epitope,conservence_presence,allel
0,ASSGL,DAASSGLTK,ASSGL,HLA-A*68:01
1,QDFSQDPSKFTQP,DPSKFTQPV,DPSKFTQP,"HLA-B*51:01,HLA-B*35:01,HLA-B*08:01,"
2,NPALM,DVLTNPALM,NPALM,HLA-A*26:01


In [169]:
final_table.to_csv(path_or_buf="result_collection/new results/VP4_CTL_final_candidate_epitope_with_allel.csv")

In [170]:
# filtered_final = tcel1_result[tcel1_result['peptide'].isin(final_table["epitope"].values)]
# filtered_final

In [171]:
# filtered_final.to_csv(path_or_buf="/home/kaderi/Feroj/thesis/HRV-C/Data_collecton/hrv-top5/result_collection/3D_CTL/3D_CTL_with_score.csv")